# Predicción del precio de venta de vehículos usados

**Proyecto Integrador — Modelos y Simulación de Sistemas I (Grupo B, Universidad de Antioquia)**
Docente: Andrés Parra · **Fase 1 — Modelo predictivo**

| | |
|---|---|
| **Integrantes** | _(completar: Nombre 1, Nombre 2, Nombre 3)_ |
| **Repositorio** | _(URL de GitHub)_ |
| **Fecha de entrega** | 15 de septiembre de 2026 |

> Este notebook es ejecutable de principio a fin sin intervención manual
> (`Kernel → Restart & Run All`). Toda decisión de preparación de datos se
> justifica en el texto y todo ajuste (imputación, codificación) se realiza
> **solo con datos de entrenamiento** mediante un `Pipeline` de scikit-learn.

## 1. Introducción

**Problema.** En el mercado de vehículos usados el precio de venta lo fija cada
vendedor de forma manual y heterogénea. Un modelo que estime el precio a partir
de las características objetivas del vehículo sirve como referencia tanto para
quien vende (fijar un precio competitivo) como para quien compra (detectar
sobreprecios).

**Objetivo del modelo.** Predecir el precio de venta (`price`, en dólares) de un
vehículo usado a partir de sus atributos: marca, año, kilometraje, tipo de
combustible, especificaciones de motor, transmisión, colores, historial de
accidentes y estado del título.

**Tipo de problema.** Aprendizaje supervisado, **regresión** (la variable
objetivo es cuantitativa continua).

**Variable objetivo.** `price` — precio de venta publicado, en dólares
estadounidenses. En el archivo original viene como texto (`"$10,300"`) y se
convierte a numérico durante la preparación.

**Fuente de datos.** *Used Car Price Prediction Dataset*, Kaggle
(https://www.kaggle.com/datasets/taeefnajib/used-car-price-prediction-dataset).
Corte transversal: cada fila es un vehículo listado en un momento dado; **no es
una serie temporal**. El archivo se versiona en el repositorio en
`fase-1/data/used_cars.csv`.

## 2. Configuración del entorno

Fijamos una semilla global (`RANDOM_STATE = 42`) que se reutiliza en la
separación train/test y en todos los estimadores con componente aleatoria, para
que el notebook sea reproducible.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

print("Versiones — pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", __import__("sklearn").__version__)

: 

In [ ]:
# El notebook funciona tanto si se ejecuta desde fase-1/ como desde la raíz del repo.
DATA_PATH = Path("data/used_cars.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("fase-1/data/used_cars.csv")

df = pd.read_csv(DATA_PATH)
print("Archivo:", DATA_PATH.resolve())
print("Dimensiones:", df.shape)
df.head()

## 3. Descripción del dataset

- **Observaciones:** 4 009 vehículos (filas).
- **Variables:** 12 columnas — 11 predictoras potenciales + la variable objetivo.
- **Unidad de observación:** un vehículo publicado. Corte transversal, sin
  componente temporal.

**Partimos de las 12 columnas originales.** Ninguna se descarta de entrada;
cualquier eliminación se justifica más adelante, durante el análisis.

| Columna | Tipo | Descripción | Observaciones iniciales |
|---|---|---|---|
| `brand` | categórica | Marca | 57 categorías |
| `model` | categórica | Modelo | **1 898 categorías** (cardinalidad muy alta) |
| `model_year` | numérica | Año de fabricación | rango 1974–2024 |
| `milage` | texto→numérica | Kilometraje | viene como `"51,000 mi."` |
| `fuel_type` | categórica | Combustible | nulos + valores corruptos (`�`, `not supported`) |
| `engine` | texto | Especificaciones del motor | texto libre; se puede extraer HP, litros, cilindros |
| `transmission` | categórica | Transmisión | 62 variantes de texto |
| `ext_col` | categórica | Color exterior | 319 categorías |
| `int_col` | categórica | Color interior | 156 categorías |
| `accident` | categórica | Historial de accidentes | binaria + nulos |
| `clean_title` | categórica | Título limpio | solo aparece el valor `"Yes"` + nulos |
| `price` | texto→numérica | **Precio de venta (objetivo)** | viene como `"$10,300"` |

**Limitaciones conocidas del dataset.**

1. No incluye ubicación geográfica ni fecha de publicación, factores que influyen
   en el precio real y que aquí quedan como ruido no explicado.
2. `price` está fuertemente sesgado a la derecha: coexisten autos de uso masivo
   con vehículos de colección de cientos de miles a millones de dólares.
3. Alta cardinalidad en `model`, `ext_col`, `int_col`.
4. Valores faltantes en `fuel_type`, `accident` y `clean_title` (se cuantifican
   en el EDA).

In [ ]:
df.info()

In [ ]:
# Estadísticos de las columnas numéricas tal como vienen (solo model_year lo es).
df.describe(include="all").T

## 4. Análisis exploratorio (EDA)

### 4.1 Valores faltantes

In [ ]:
nulos = pd.DataFrame({
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2),
})
nulos = nulos.sort_values("n_nulos", ascending=False)
display(nulos)

ax = nulos[nulos.n_nulos > 0]["pct_nulos"].plot.bar(figsize=(6, 3.5), color="#4C72B0")
ax.set_ylabel("% de valores faltantes")
ax.set_title("Valores faltantes por variable")
plt.tight_layout()
plt.show()

Solo tres variables tienen faltantes:

| Variable | % faltante | Lectura |
|---|---|---|
| `clean_title` | ~14.9 % | La columna **solo contiene el valor `"Yes"`**; el faltante actúa como "no consta título limpio". |
| `fuel_type` | ~4.2 % | Además hay 45 registros con el carácter corrupto `�` y 2 con `"not supported"` que también son, en la práctica, faltantes. |
| `accident` | ~2.8 % | Binaria (`None reported` / `At least 1 accident...`); el faltante es "sin información". |

El curso recomienda faltantes en el rango 0.1 %–2 %, pero admite porcentajes
mayores si están justificados por el contexto. Aquí lo están: son campos que el
publicador puede dejar vacíos, y el propio hecho de estar vacío es informativo
(por eso los trataremos como categoría `"Unknown"` y no eliminando filas).

### 4.2 Limpieza determinista mínima para poder explorar

`price` y `milage` son numéricas disfrazadas de texto. Convertirlas quitando
`$`, `,` y `" mi."` es una operación **determinista fila a fila**: no aprende
ningún parámetro de los datos, así que puede hacerse antes de separar train/test
sin generar fuga de información. Lo mismo aplica a extraer HP / litros / cilindros
de `engine` y el número de marchas de `transmission` con expresiones regulares.

Construimos un `df_eda` auxiliar solo para las gráficas de esta sección; la
preparación formal para el modelo se define en la sección 5.

In [ ]:
def parse_money(serie):
    ''''$10,300' -> 10300.0 ; '51,000 mi.' -> 51000.0. Determinista, sin ajuste.'''
    return pd.to_numeric(
        serie.astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce"
    )

df_eda = df.copy()
df_eda["price"] = parse_money(df_eda["price"])
df_eda["milage_num"] = parse_money(df_eda["milage"])
df_eda["antiguedad"] = 2024 - df_eda["model_year"]
_eng = df_eda["engine"].astype(str)
df_eda["engine_hp"] = pd.to_numeric(_eng.str.extract(r"([\d.]+)\s*HP", flags=re.I)[0], errors="coerce")
df_eda["engine_liters"] = pd.to_numeric(_eng.str.extract(r"([\d.]+)\s*L", flags=re.I)[0], errors="coerce")
df_eda["engine_cylinders"] = pd.to_numeric(_eng.str.extract(r"(\d+)\s*Cylinder", flags=re.I)[0], errors="coerce")

df_eda[["price", "milage_num", "antiguedad", "engine_hp", "engine_liters", "engine_cylinders"]].describe().T

### 4.3 Distribución de la variable objetivo (`price`)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(df_eda["price"], bins=60, color="#4C72B0")
axes[0].set_title("price (escala original)")
axes[0].set_xlabel("USD")

axes[1].boxplot(df_eda["price"], vert=True)
axes[1].set_title("price — caja (outliers de gama alta)")

axes[2].hist(np.log1p(df_eda["price"]), bins=60, color="#55A868")
axes[2].set_title("log1p(price)")
axes[2].set_xlabel("log(USD)")
plt.tight_layout()
plt.show()

print(df_eda["price"].describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(0))
print("\nAsimetría (skew)  price      :", round(df_eda["price"].skew(), 2))
print("Asimetría (skew)  log1p(price):", round(np.log1p(df_eda["price"]).skew(), 2))
print("Vehículos con price > 200 000 USD:", int((df_eda["price"] > 200_000).sum()),
      f"({(df_eda['price'] > 200_000).mean()*100:.1f} %)")

`price` tiene asimetría ≈ 19.5: el 99 % de los autos vale menos de ~273 000 USD,
pero hay ejemplares de colección (Bugatti, Rolls-Royce, Maserati) que llegan a
~3 000 000 USD. **No son errores de captura**, son vehículos reales, así que no
se eliminan; pero distorsionan cualquier métrica basada en el error al cuadrado.

**Decisión:** entrenar sobre `log1p(price)` (asimetría ≈ 0.1, casi simétrica) y
volver a la escala de dólares para reportar. Esto se implementa con
`TransformedTargetRegressor`, que aplica `log1p` antes de entrenar y `expm1` a
las predicciones de forma automática.

### 4.4 Relación de las predictoras con `price`

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].scatter(df_eda["milage_num"], df_eda["price"], s=6, alpha=.3)
axes[0, 0].set(title="price vs kilometraje", xlabel="millas", ylabel="USD", yscale="log")

axes[0, 1].scatter(df_eda["antiguedad"], df_eda["price"], s=6, alpha=.3, color="#C44E52")
axes[0, 1].set(title="price vs antigüedad", xlabel="años (2024 - model_year)", ylabel="USD", yscale="log")

top_brands = df_eda.groupby("brand")["price"].median().sort_values(ascending=False).head(12)
top_brands.plot.barh(ax=axes[1, 0], color="#4C72B0")
axes[1, 0].set(title="Mediana de price por marca (top 12)", xlabel="USD")
axes[1, 0].invert_yaxis()

sns.boxplot(data=df_eda, x="accident", y="price", ax=axes[1, 1])
axes[1, 1].set(title="price por historial de accidentes", yscale="log")
axes[1, 1].set_xticklabels(["None reported", "≥1 accidente", "(NaN)"], rotation=10)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.boxplot(data=df_eda, x="fuel_type", y="price", ax=ax)
ax.set(title="price por tipo de combustible", yscale="log")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

Lecturas del EDA bivariado:

- **Kilometraje**: relación negativa clara con el precio (más uso → menor valor).
- **Antigüedad**: relación negativa, aunque los autos muy antiguos (clásicos)
  rompen la tendencia y suben de precio.
- **Marca**: separa con fuerza el precio (Rolls-Royce, Ferrari, Lamborghini,
  Porsche en la parte alta). Es una predictora valiosa.
- **Accidentes**: los vehículos sin accidentes reportados tienen mediana mayor.
- **Combustible**: los híbridos enchufables y diésel muestran medianas algo más
  altas; `Gasoline` domina en volumen.

### 4.5 Matriz de correlación (variables numéricas)

In [ ]:
num_cols = ["price", "milage_num", "antiguedad", "engine_hp", "engine_liters", "engine_cylinders"]
corr = df_eda[num_cols].corr()

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación de Pearson")
plt.tight_layout()
plt.show()

- `engine_hp` y `engine_liters` son las numéricas más correlacionadas con
  `price` (a más potencia/cilindrada, más precio).
- `milage_num` y `antiguedad` correlacionan **positivamente entre sí** (autos más
  viejos acumulan más millas) y negativamente con el precio.
- `engine_hp`–`engine_liters` están correlacionadas entre sí (~0.7). No es un
  problema para un modelo de árboles como el que usaremos; se menciona por
  transparencia.

### 4.6 Cardinalidad de las variables categóricas

In [ ]:
cat_cols = ["brand", "model", "fuel_type", "transmission", "ext_col", "int_col",
            "accident", "clean_title"]
card = pd.DataFrame({
    "n_categorias": df[cat_cols].nunique(),
    "moda": [df[c].mode(dropna=True).iat[0] for c in cat_cols],
    "frec_moda_%": [(df[c].value_counts(normalize=True).iat[0] * 100).round(1) for c in cat_cols],
})
display(card.sort_values("n_categorias", ascending=False))

print("Ejemplos de 'model':", df["model"].value_counts().head(5).to_dict())
print("Registros por valor de 'model' (media):", round(len(df) / df["model"].nunique(), 2))

`model` tiene 1 898 categorías para 4 009 filas: en promedio **~2 autos por
modelo**. Codificarla (one-hot o de destino) produciría cientos de columnas casi
vacías o memorización del precio por grupo. **Decisión: se descarta `model`.**
La marca (`brand`, 57 categorías) captura buena parte de la señal de segmento y
sí es codificable con garantías. Se deja documentado que en fases posteriores
podría reincorporarse con *target encoding* validado por CV.

`ext_col` (319) e `int_col` (156) se conservan pero agrupando las categorías
raras (frecuencia < 1 %) en `"infrequent"` mediante `OneHotEncoder`, ajustado
**solo con train**.

## 5. Preparación de datos — decisiones y justificaciones

| Variable original | Tratamiento | Justificación |
|---|---|---|
| `price` | `parse_money` → numérico; objetivo en `log1p` | quitar `$`/`,`; corregir asimetría extrema |
| `milage` | `parse_money` → `milage_num` | quitar `,` y `" mi."` |
| `model_year` | → `antiguedad = 2024 - model_year` | 2024 es el año más reciente del dataset y aproxima la fecha de publicación; la antigüedad es más interpretable. Se elimina `model_year` (colineal exacto). |
| `engine` | regex → `engine_hp`, `engine_liters`, `engine_cylinders` + indicadores de faltante | convierte texto libre en señal numérica útil (potencia/cilindrada pesan en el precio) |
| `transmission` | regex → `transmission_speeds` (nº de marchas) y `transmission_type` ∈ {Automatic, Manual, CVT, Other} | reduce 62 variantes de texto a 2 features con sentido mecánico |
| `fuel_type` | valores fuera de un conjunto conocido (incl. `�`, `not supported`, NaN) → `"Unknown"` | son faltantes disfrazados |
| `accident` | NaN → `"Unknown"` | el faltante es informativo; no se borran filas |
| `clean_title` | `== "Yes"` → binaria `has_clean_title` (1/0) | la columna solo tiene el valor `"Yes"`; el resto es "no consta" |
| `model` | **se descarta** | 1 898 categorías (~2 filas/categoría): riesgo de sobreajuste / fuga por grupo |
| `ext_col`, `int_col` | `OneHotEncoder(min_frequency=0.01, handle_unknown="infrequent_if_exist")` | agrupa colores raros; robusto ante categorías nuevas en test |
| `brand`, `fuel_type_clean`, `accident_clean`, `transmission_type` | `OneHotEncoder(handle_unknown="ignore")` | baja cardinalidad |
| numéricas | `SimpleImputer(strategy="median", add_indicator=True)` | imputación robusta + marca de "faltaba" |

**Regla anti-fuga:** todos los `SimpleImputer` y `OneHotEncoder` se ajustan
**dentro de un `Pipeline`** que solo ve `X_train`. El conjunto de prueba nunca
interviene en `fit`.

### 5.1 Definición de `X`, `y` y función de limpieza

In [ ]:
REF_YEAR = int(df["model_year"].max())          # 2024
KNOWN_FUEL = {"Gasoline", "Hybrid", "E85 Flex Fuel", "Diesel", "Plug-In Hybrid"}
COLS_DESCARTADAS = ["model", "model_year", "milage", "engine", "transmission",
                    "fuel_type", "accident", "clean_title", "price"]

def clean_features(data: pd.DataFrame) -> pd.DataFrame:
    '''Transformación determinista fila a fila (sin parámetros ajustados).

    Recibe el DataFrame crudo (sin 'price') y devuelve las columnas de entrada
    listas para el preprocesador. Al no aprender nada de los datos, es seguro
    aplicarla antes del split.
    '''
    out = pd.DataFrame(index=data.index)
    out["milage_num"] = parse_money(data["milage"])
    out["antiguedad"] = REF_YEAR - pd.to_numeric(data["model_year"], errors="coerce")

    eng = data["engine"].astype(str)
    out["engine_hp"] = pd.to_numeric(eng.str.extract(r"([\d.]+)\s*HP", flags=re.I)[0], errors="coerce")
    out["engine_liters"] = pd.to_numeric(eng.str.extract(r"([\d.]+)\s*L", flags=re.I)[0], errors="coerce")
    out["engine_cylinders"] = pd.to_numeric(eng.str.extract(r"(\d+)\s*Cylinder", flags=re.I)[0], errors="coerce")

    trans = data["transmission"].astype(str)
    out["transmission_speeds"] = pd.to_numeric(trans.str.extract(r"(\d+)-Speed", flags=re.I)[0], errors="coerce")
    t = trans.str.lower()
    out["transmission_type"] = np.select(
        [t.str.contains("cvt"), t.str.contains(r"m/t|manual"), t.str.contains(r"a/t|automatic|dual shift")],
        ["CVT", "Manual", "Automatic"], default="Other",
    )

    out["brand"] = data["brand"].astype(str)
    out["fuel_type_clean"] = data["fuel_type"].where(data["fuel_type"].isin(KNOWN_FUEL), "Unknown").astype(str)
    out["accident_clean"] = data["accident"].fillna("Unknown").astype(str)
    out["has_clean_title"] = (data["clean_title"] == "Yes").astype("int64")
    out["ext_col"] = data["ext_col"].astype(str)
    out["int_col"] = data["int_col"].astype(str)
    return out

y = parse_money(df["price"])
X = df.drop(columns=["price"])

assert "price" not in X.columns, "La variable objetivo no debe estar entre las predictoras"
print("REF_YEAR =", REF_YEAR)
print("X:", X.shape, "| y:", y.shape, "| nulos en y:", int(y.isna().sum()))
clean_features(X).head()

### 5.2 Separación train / test

- **Método:** `train_test_split` aleatorio, **80 % entrenamiento / 20 % prueba**,
  `random_state=42`.
- **Riesgo de fuga por grupos naturales.** Un mismo `model` (o `brand`) puede
  aparecer en train y en test. Como **`model` se descarta** y `brand` es una
  característica legítima de segmento (no un identificador de fila), no hay fuga
  de identidad: el modelo aprende "los Porsche valen más", no "esta fila concreta
  vale X". Se documenta la alternativa `GroupShuffleSplit(groups=df['model'])`
  para fases posteriores si se decidiera reincorporar `model`.
- No hay variables "del futuro": todas las predictoras describen el vehículo en
  el momento de la publicación, no información posterior a la venta.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("train:", X_train.shape, " test:", X_test.shape)

solape = set(X_test["brand"]).difference(X_train["brand"])
print("Marcas presentes en test y ausentes en train:", solape or "ninguna")
print("Proporción price>200k  train:",
      f"{(y_train > 200_000).mean()*100:.1f} %  test: {(y_test > 200_000).mean()*100:.1f} %")

### 5.3 Pipeline de preprocesamiento

`ColumnTransformer` aplica a cada grupo de columnas su tratamiento. Se ajusta
**solo con `X_train`** (dentro del pipeline del modelo), por lo que medianas y
categorías se calculan sin ver el test.

In [ ]:
NUM_FEATURES = ["milage_num", "antiguedad", "engine_hp", "engine_liters",
                "engine_cylinders", "transmission_speeds", "has_clean_title"]
CAT_LOW = ["brand", "fuel_type_clean", "accident_clean", "transmission_type"]
CAT_HIGH = ["ext_col", "int_col"]

preprocesador = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median", add_indicator=True), NUM_FEATURES),
        ("cat_low", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_LOW),
        ("cat_high", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=0.01, sparse_output=False)),
        ]), CAT_HIGH),
    ],
    remainder="drop",
)
preprocesador

## 6. Prevención de fuga de información

Acciones tomadas y verificadas explícitamente:

1. **`price` fuera de las predictoras.** `X = df.drop(columns=["price"])` y un
   `assert` lo confirma. La única transformación de `price` es a la variable
   objetivo `y`.
2. **El test no interviene en el entrenamiento.** El split se hace *antes* de
   cualquier `fit`. Imputación y one-hot viven dentro del `Pipeline`, que solo
   recibe `X_train` en `fit`. `cross_val_score` y `GridSearch` (fases futuras)
   operan solo sobre train.
3. **La limpieza previa al split no ajusta parámetros.** `clean_features` y
   `parse_money` son deterministas fila a fila (regex, aritmética, comparaciones
   contra constantes definidas a mano). Se verifica abajo que aplicarlas sobre
   train y sobre todo `X` da exactamente el mismo resultado para las filas de
   train (idempotencia / independencia entre filas).
4. **Sin variables del futuro.** Todas las predictoras son atributos del vehículo
   conocidos al momento de publicarlo; ninguna se deriva del precio ni de eventos
   posteriores a la venta.

In [ ]:
# (1) price no está entre las predictoras
assert "price" not in X.columns and "price" not in X_train.columns

# (3) clean_features es independiente entre filas: el resultado para las filas de
#     train no cambia si se calcula sobre todo X en vez de solo sobre X_train.
ct_solo_train = clean_features(X_train)
ct_todo = clean_features(X).loc[X_train.index]
assert ct_solo_train.equals(ct_todo), "clean_features dependería del conjunto -> fuga"

# (4) columnas efectivamente usadas como entrada
print("Columnas de entrada al modelo :", NUM_FEATURES + CAT_LOW + CAT_HIGH)
print("Columnas descartadas          :", COLS_DESCARTADAS)
print("\nVerificaciones anti-fuga: OK")

## 7. Modelo base (baseline)

`DummyRegressor(strategy="median")` predice siempre la **mediana** del precio de
entrenamiento. Es el punto de comparación obligatorio: cualquier modelo útil debe
superarlo con claridad. Se elige la mediana (y no la media) porque es robusta al
sesgo extremo de `price`.

In [ ]:
def evaluar(nombre, y_true, y_pred, extra_subset=True):
    '''Devuelve una fila con MAE, RMSE y R2 en dólares. Opcionalmente añade
    las mismas métricas para el mercado masivo (price < 200k USD).'''
    fila = {
        "modelo": nombre,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }
    if extra_subset:
        m = np.asarray(y_true) < 200_000
        fila["MAE_<200k"] = mean_absolute_error(np.asarray(y_true)[m], np.asarray(y_pred)[m])
        fila["R2_<200k"] = r2_score(np.asarray(y_true)[m], np.asarray(y_pred)[m])
    return fila

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

resultados = [evaluar("Baseline (mediana)", y_test, pred_baseline)]
pd.DataFrame(resultados).round(3)

## 8. Modelo predictivo — Random Forest

**Algoritmo elegido:** `RandomForestRegressor`.

Justificación:

- Captura relaciones **no lineales** (precio vs antigüedad no es monótona) e
  **interacciones** (marca × potencia) sin ingeniería adicional.
- Es **robusto a outliers** y a escalas distintas entre variables (no requiere
  estandarizar).
- Pocos hiperparámetros sensibles; da un resultado sólido casi "de fábrica".
- Ofrece **importancia de variables**, útil para interpretar el modelo.

El objetivo se modela en escala logarítmica con `TransformedTargetRegressor`
(`func=log1p`, `inverse_func=expm1`): el bosque entrena sobre `log1p(price)` y las
predicciones se devuelven ya en dólares.

In [ ]:
def construir_modelo(estimador):
    '''clean_features -> preprocesador -> estimador, con objetivo en log1p.'''
    pipe = Pipeline([
        ("clean", FunctionTransformer(clean_features)),
        ("pre", preprocesador),
        ("model", estimador),
    ])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

rf = construir_modelo(RandomForestRegressor(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
))
rf.fit(X_train, y_train)

# Validación cruzada (5 folds) SOLO sobre entrenamiento — sin tocar el test.
cv_mae = -cross_val_score(rf, X_train, y_train, cv=5,
                          scoring="neg_mean_absolute_error", n_jobs=-1)
print(f"MAE en CV (train, 5 folds): {cv_mae.mean():,.0f} USD  ±{cv_mae.std():,.0f}")

pred_rf = rf.predict(X_test)
resultados.append(evaluar("Random Forest", y_test, pred_rf))
pd.DataFrame(resultados).round(3)

### 8.1 Comparación con un segundo algoritmo (Gradient Boosting)

No es obligatorio comparar varios modelos, pero un `HistGradientBoostingRegressor`
sirve de contraste rápido y confirma que el Random Forest está en un rango
razonable.

In [ ]:
hgb = construir_modelo(HistGradientBoostingRegressor(
    random_state=RANDOM_STATE, max_iter=400, learning_rate=0.05
))
hgb.fit(X_train, y_train)
pred_hgb = hgb.predict(X_test)
resultados.append(evaluar("Hist Gradient Boosting", y_test, pred_hgb))
pd.DataFrame(resultados).round(3)

## 9. Evaluación en el conjunto de prueba

**Métricas elegidas** (todas en dólares, sobre el 20 % de test que el modelo
nunca vio):

- **MAE** — error absoluto medio. Métrica **principal**: está en dólares, es
  interpretable y es robusta a los pocos vehículos de gama millonaria.
- **RMSE** — penaliza más los errores grandes; muy afectada por los exóticos.
- **R²** — proporción de varianza explicada.

Además reportamos MAE y R² restringidos al **mercado masivo** (`price < 200 000`,
el 98 % de las filas) para separar el desempeño típico del efecto de los
outliers.

In [ ]:
tabla = pd.DataFrame(resultados).set_index("modelo").round(3)
display(tabla)

mae_base = tabla.loc["Baseline (mediana)", "MAE"]
mae_rf = tabla.loc["Random Forest", "MAE"]
print(f"MAE baseline      : {mae_base:,.0f} USD")
print(f"MAE Random Forest : {mae_rf:,.0f} USD")
print(f"Reducción del MAE : {(1 - mae_rf / mae_base) * 100:.1f} %  respecto al baseline")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].scatter(y_test, pred_rf, s=10, alpha=.35)
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set(xscale="log", yscale="log", xlabel="price real (USD)",
           ylabel="price predicho (USD)", title="Predicho vs. real (Random Forest)")

resid = y_test.values - pred_rf
axes[1].hist(resid[np.abs(resid) < 60_000], bins=60, color="#4C72B0")
axes[1].set(title="Residuos (|resid| < 60k)", xlabel="real - predicho (USD)")

importancias = pd.Series(
    rf.regressor_.named_steps["model"].feature_importances_,
    index=rf.regressor_.named_steps["pre"].get_feature_names_out(),
).sort_values(ascending=False).head(12)
importancias.iloc[::-1].plot.barh(ax=axes[2], color="#55A868")
axes[2].set(title="Top-12 importancia de variables (RF)")

plt.tight_layout()
plt.show()

### 9.1 Discusión

**¿Mejora al baseline?** Sí, con claridad. El MAE baja de ~32 900 USD (baseline)
a ~17 000 USD con Random Forest: alrededor de un **48 % menos de error**. En el
mercado masivo (`price < 200k`) el MAE del Random Forest es de ~7 300 USD y el
R² ≈ 0.84.

**¿La métrica es razonable?** Depende del segmento:

- Para el 98 % de los vehículos (uso masivo) el error típico de ~7 000 USD sobre
  precios de 15 000–50 000 USD es un resultado útil como referencia.
- El **R² global (~0.10)** es engañosamente bajo: lo hunden ~17 autos de test que
  valen entre 200 000 y 3 000 000 USD. Sobre `log1p(price)` el R² del modelo es
  ~0.82, y sobre el mercado masivo ~0.84. Es un problema de **métrica sensible a
  outliers**, no de un modelo que "no aprende".

**Dificultades encontradas.**

- Sesgo extremo de `price` (resuelto parcialmente con la transformación log).
- Texto libre en `engine`/`transmission` (resuelto con regex; ~20 % de HP queda
  sin extraer y se imputa).
- Alta cardinalidad en `model` y colores (resuelta descartando `model` y
  agrupando colores raros).
- Faltantes informativos en `accident`/`clean_title` (tratados como categoría).

**¿Qué mejoraríamos?**

- Segmentar el problema (un modelo para uso masivo y otro para gama alta) o usar
  una pérdida robusta / cuantílica.
- Ajuste de hiperparámetros con `GridSearchCV`/`RandomizedSearchCV` sobre train.
- Reincorporar `model` mediante *target encoding* validado por CV.
- Enriquecer `engine` (tracción, turbo) y añadir interacciones marca×antigüedad.

## 10. Reproducibilidad

- Semilla única `RANDOM_STATE = 42` en `np.random.seed`, `train_test_split` y
  todos los estimadores aleatorios.
- El notebook se ejecuta de arriba abajo sin intervención manual y sin celdas de
  prueba sueltas.
- Versiones de librerías impresas en la sección 2; dependencias congeladas en
  `requirements.txt`.
- El dataset está versionado en el repositorio (`fase-1/data/used_cars.csv`).

## 11. Persistencia del modelo

Guardamos el pipeline **completo** (limpieza + preprocesamiento + modelo +
transformación del objetivo) en `modelo.joblib`. Así el artefacto recibe un
`DataFrame` con las columnas crudas del dataset (sin `price`) y devuelve el
precio en dólares, sin pasos externos.

Para la entrega se usa una versión ligeramente podada del bosque
(`n_estimators=200`, `min_samples_leaf=2`) y compresión `joblib`, de modo que el
`.joblib` pese pocos MB y sea cómodo de versionar. El desempeño es equivalente
(MAE de mercado masivo ~7 700 USD).

> En la Fase 2, `clean_features` y `parse_money` se moverán a un módulo importable
> (`src/`) para que `predict.py` cargue el modelo sin depender de este notebook.

In [ ]:
modelo_final = construir_modelo(RandomForestRegressor(
    n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
))
modelo_final.fit(X_train, y_train)

RUTA_MODELO = Path("modelo.joblib")
if not Path("data").exists():          # ejecutado desde la raíz del repo
    RUTA_MODELO = Path("fase-1/modelo.joblib")

joblib.dump(modelo_final, RUTA_MODELO, compress=3)
print(f"Modelo guardado en {RUTA_MODELO.resolve()}  ({RUTA_MODELO.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Demostración: recargar el modelo desde disco y predecir con el objeto cargado.
modelo_cargado = joblib.load(RUTA_MODELO)

muestra = X_test.head(5)
comparacion = pd.DataFrame({
    "price_real": y_test.head(5).values,
    "price_predicho": modelo_cargado.predict(muestra).round(0),
})
comparacion["error_abs"] = (comparacion.price_real - comparacion.price_predicho).abs()
display(comparacion)

# El modelo recargado produce exactamente las mismas predicciones que el original.
np.testing.assert_allclose(
    modelo_cargado.predict(X_test), modelo_final.predict(X_test), rtol=1e-9
)
print("El modelo recargado reproduce las predicciones del original: OK")

## 12. Conclusiones

- Se construyó un modelo de **regresión** que predice el precio de vehículos
  usados a partir de sus características, superando al baseline en ~48 % de MAE.
- La preparación de datos se justificó variable por variable y se implementó
  **sin fuga de información**: todo ajuste ocurre dentro de un `Pipeline` que solo
  ve el entrenamiento; `price` nunca está entre las predictoras.
- La principal limitación es el sesgo extremo de `price`; el modelo es fiable en
  el mercado masivo (R² ≈ 0.84) y débil en la gama de colección.
- El modelo se persiste con `joblib` y se verifica que puede recargarse y
  predecir. Este pipeline es la base de la Fase 2 (scripts + Docker).